# Single-Cell RNA Sequencing Profiles, Variant Annotations, and Genomic Features from Human Peripheral Blood Mononuclear Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.0xyk-jqc8/fair2.json`

**Note**: All entities (record sets, fields, columns) are referenced by their `@id` as per Croissant standards.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.0xyk-jqc8/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Name: {metadata.name}\n\nDescription: {metadata.description}\n")
print(f"Published: {metadata.datePublished}")
print(f"Authors (@id): {[author['@id'] for author in metadata.author]}")
print(f"Keywords: {metadata.keywords}")

## 2. Data Overview
Review available record sets, fields, columns, and their IDs.

**Note:** The Croissant schema organizes data by record sets (`cr:RecordSet`). We will enumerate the record sets, their fields, and columns using their `@id` values.

In [ ]:
# List available record sets by their @id

record_sets = dataset.metadata.recordSet if hasattr(dataset.metadata, 'recordSet') else []
if not record_sets:
    print("No record sets found in metadata.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        # List fields and columns
        fields = rs.get('field', [])
        columns = rs.get('column', [])
        if fields:
            print("  Fields:")
            for f in fields:
                print(f"    - {f['@id']} ({f.get('name', 'N/A')})")
        if columns:
            print("  Columns:")
            for c in columns:
                print(f"    - {c['@id']} ({c.get('name', 'N/A')})")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Below, we demonstrate loading all available record sets and listing their columns, referencing by `@id`.

In [ ]:
# Prepare record sets' @id list for extraction
if not record_sets:
    print("No record sets available in the metadata.")
else:
    record_sets_ids = [rs['@id'] for rs in record_sets]
    dataframes = {}
    for rs_id in record_sets_ids:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} records for RecordSet @id: {rs_id}")
            print("Columns:", df.columns.tolist())
            print(df.head(3), '\n')
        else:
            print(f"No records found for RecordSet @id: {rs_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below, we select a record set, identify a numeric field, apply filtering, normalization, and grouping. All fields are referenced by their `@id`.

In [ ]:
# Assuming at least one record set and at least one numeric field...

if dataframes:
    # Choose the first record set for demonstration
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

    # Try to automatically detect a numeric field
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

    if numeric_field_id is not None:
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Find a proper group field (categorical/string field)
        group_field_id = None
        for col in df.columns:
            if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id:
                group_field_id = col
                break

        if group_field_id is not None:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric field found in the selected record set.")
else:
    print("No dataframes loaded to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We use matplotlib to plot numeric data distributions by group (if available).

In [ ]:
import matplotlib.pyplot as plt

if dataframes and 'numeric_field_id' in locals() and numeric_field_id is not None:
    plt.figure(figsize=(8,5))
    filtered_df[numeric_field_id].hist(bins=30)
    plt.title(f"Distribution of {numeric_field_id} in filtered records")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if 'group_field_id' in locals() and group_field_id is not None:
        grouped_means = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        grouped_means.plot(kind='bar', figsize=(10,5))
        plt.title(f"Mean {numeric_field_id} by {group_field_id} (filtered records)")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field_id)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load, examine, process, and visualize the FAIR^2 dataset using `mlcroissant`.

- All dataset entities were referenced by their Croissant `@id`.
- The workflow included data loading, overview, extraction, processing, and visualization.

Further steps could include advanced analyses, machine learning, or integration with downstream bioinformatics pipelines.